# This notebook is to sort by channel and then cluster each channel

**USE Cleaned Data!**

In [1]:
from treeview_gen import df_to_treeview
import pandas as pd
import os
import re
import shutil
from time import sleep

regex_df = pd.read_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed/HistDiff_results_24h_bmal1ref_REDONE_WELL-DROPPED_FeatRed.csv", index_col=0, nrows=1)
pattern = r'Region (.+?) \(global\)'
channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)


{'TMR/AF555 Rabbitts', 'HOECHST 33342 expanded', 'MitoTracker Deep Red', 'SYTO14 Rabbitts', 'Alexa 488 shortened'}


In [2]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str, sortIndex: bool = True) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=sortIndex, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

In [ ]:
test_cdt = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED/Alexa 488 shortened/Alexa 488 shortened_treeview.cdt"
cdt_df = pd.read_csv(test_cdt, sep="\t", index_col=[0,1,2])

print(cdt_df.columns.tolist())

## Final Loop

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    print(curr_file)
    raw_name = os.path.splitext(file)[0]
    print(raw_name)
    
    raw_out_dir = os.path.join(out_dir, raw_name)
    print(raw_out_dir)
    
    # make a folder
    if os.path.exists(raw_out_dir):
        shutil.rmtree(raw_out_dir)
        sleep(0.5)
    os.makedirs(raw_out_dir)
    
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    
    # grab channels from data
    channel_df = {ch: curr_df[[col for col in curr_df.columns if ch in col]] for ch in channels_found}
    
    # grab the column orders and export out the channel clustering
    col_order = []
    for name, df in channel_df.items():
        order = gen_treeview_channel(df, name.replace("/", "_"), raw_out_dir)
        col_order.extend(order)
    
    print(col_order)
    
    reorder_df = curr_df[col_order]
    
    final_out_path = os.path.join(raw_out_dir, f"{raw_name}_SORTED-CLUSTERED_channels")
    df_to_treeview(df=reorder_df, outFile=final_out_path, sortIndex=True, row_cluster=False, col_cluster=False)

# We made an error:

- **ALL HEATMAPS MUST HAVE THE SAME COLUMN ORDERINGS AND CLUSTERING**
- This means we need to do the following:
1. Combine all the datasets together
2. Split channels
3. cluster columns of combined but channel split dataset
4. retrieve ordering
5. sort ids and apply ordering

In [3]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

In [4]:
# lets remove the orginal channel sorted folder
def clean_folder(path: str) -> None:
    if os.path.exists(path):
        shutil.rmtree(path)
        sleep(0.5)
    os.makedirs(path, exist_ok=True)

In [5]:
clean_folder(out_dir)

In [5]:
# combine data sets
renamed_datasets = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

dfs = []
for file in os.listdir(renamed_datasets):
    curr_file = os.path.join(renamed_datasets, file)
    curr_df = pd.read_csv(curr_file, index_col=0)
    dfs.append(curr_df)

concat_df = pd.concat(dfs)
display(concat_df)
print(concat_df.shape)

concat_df.to_csv('/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_data.csv')

,Nuclei - Cell Region Alexa 488 shortened (global) Profile 2/5 SER-Hole,Nuclei - Ring Region HOECHST 33342 expanded (global) Radial Relative Deviation SER-Ridge,Nuclei - Ring Region SYTO14 Rabbitts (global) Profile 2/5 SER-Hole,Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 3/5 SER-Bright,Nuclei - Nucleus Region SYTO14 Rabbitts (global) Radial Mean Ratio SER-Valley,Nuclei - Intensity Membrane Region MitoTracker Deep Red (global) Contrast,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 30%,Nuclei - Cell Region MitoTracker Deep Red (global) Profile 2/5,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 13 SER-Bright,Nuclei - Ring Region Alexa 488 shortened (global) Profile 1/5,...,Nuclei - Nucleus Region HOECHST 33342 expanded (global) Symmetry 05 SER-Bright,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 1/5,Nuclei - Cytoplasm Region MitoTracker Deep Red (global) Threshold Compactness 40%,Nuclei - Membrane Region MitoTracker Deep Red (global) Threshold Compactness 30% SER-Edge,Nuclei - Ring Region HOECHST 33342 expanded (global) Gabor Min 2 px w2,Nuclei - Ring Region MitoTracker Deep Red (global) Radial Mean Ratio SER-Edge,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Threshold Compactness 30%,Nuclei - Cell Region HOECHST 33342 expanded (global) Threshold Compactness 30%,Nuclei - Intensity Ring Region HOECHST 33342 expanded (global) StdDev,Nuclei - Membrane Region HOECHST 33342 expanded (global) Profile 2/5 SER-Hole
human_readable_ids,,,,,,,,,,,,,,,,,,,,,
no_treatment_bmal1._.A10._.24h._.bmal1ref,0.002570,-0.000127,0.000008,-0.000064,0.000220,-0.000272,-0.000084,-0.000118,-0.000523,0.000083,...,-0.000169,0.000184,0.000054,-0.000040,1.205608e-04,-0.000057,-0.000042,-0.000877,0.003708,-1.190946e-05
no_treatment_bmal1._.A11._.24h._.bmal1ref,0.000060,0.000244,-0.000007,-0.000014,-0.000093,-0.001290,0.000673,0.000098,-0.000067,-0.000018,...,-0.000095,0.000099,0.000093,-0.000161,-5.480439e-06,0.000100,0.000214,0.000330,0.002808,-8.364908e-06
no_treatment_bmal1._.A12._.24h._.bmal1ref,-0.001167,-0.000180,-0.000007,-0.001610,-0.000058,0.001164,0.000586,0.000199,-0.000086,0.000102,...,-0.000194,-0.000259,-0.000050,-0.000040,-4.587930e-04,0.000036,0.000293,-0.000130,0.001278,-9.000905e-07
no_treatment_bmal1._.A13._.24h._.bmal1ref,0.000782,0.000297,-0.000051,-0.000051,0.000504,-0.001473,-0.000314,-0.000191,0.000255,-0.000177,...,0.000504,0.000427,-0.000115,0.000018,-3.523404e-07,0.000231,-0.000153,0.001274,0.001728,2.906337e-06
no_treatment_bmal1._.A14._.24h._.bmal1ref,-0.008598,-0.000412,0.000001,-0.000047,-0.000597,0.019484,-0.000103,0.001361,0.001008,0.002145,...,-0.000221,-0.000301,0.000267,0.001576,2.909721e-05,-0.001983,0.000151,-0.002635,0.001123,-9.838245e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PF_per2._.O05._.72h._.per2ref,-0.000008,0.000849,-0.000005,-0.001232,0.000042,0.019044,0.001375,0.000298,0.000576,0.000179,...,0.000281,-0.000183,0.001308,0.001528,-1.767191e-02,-0.001893,-0.000465,-0.008355,0.001306,1.373676e-06
PF_per2._.O06._.72h._.per2ref,-0.000020,0.001391,-0.000001,-0.004535,0.001633,0.005978,-0.000233,-0.000078,0.000216,0.000025,...,0.000272,0.000068,0.000205,0.001067,-3.699737e-03,-0.001330,-0.000802,-0.008293,0.000729,-1.046436e-06
PF_per2._.O07._.72h._.per2ref,0.000038,0.000798,-0.000023,-0.001648,0.000049,0.003919,0.000313,-0.000183,0.000989,0.000023,...,0.000189,0.000418,0.000600,0.000867,-2.904295e-02,-0.000786,-0.000855,-0.003459,0.000625,1.727853e-06


(1056, 879)


In [10]:
# sort, cluster, and grab/create final row ordering

channel_df = {ch: concat_df[[col for col in concat_df.columns if ch in col]] for ch in channels_found}

channel_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/"

# if os.path.exists(channel_out_dir):
#     shutil.rmtree(channel_out_dir)
#     sleep(0.5)
# os.makedirs(channel_out_dir, exist_ok=True)

for df in channel_df.values():
    print(df.index.tolist())

final_order = list()
for name, df in channel_df.items():
    print(name)
    print(f"Shape: {df.shape}\n")
    
    fix_name = name.replace("/", "_")
    # curr_channel_out = os.path.join(channel_out_dir, fix_name)
    # clean_folder(curr_channel_out)
    
    # file_out = os.path.join(curr_channel_out, f"{fix_name}_CHANNEL-SORTED")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir, sortIndex=False)
    final_order.extend(order)

    
print(len(final_order))
print(final_order)    
    
    

['no_treatment_bmal1._.A10._.24h._.bmal1ref', 'no_treatment_bmal1._.A11._.24h._.bmal1ref', 'no_treatment_bmal1._.A12._.24h._.bmal1ref', 'no_treatment_bmal1._.A13._.24h._.bmal1ref', 'no_treatment_bmal1._.A14._.24h._.bmal1ref', 'no_treatment_bmal1._.A15._.24h._.bmal1ref', 'no_treatment_bmal1._.A16._.24h._.bmal1ref', 'no_treatment_bmal1._.A17._.24h._.bmal1ref', 'no_treatment_bmal1._.A18._.24h._.bmal1ref', 'no_treatment_bmal1._.A19._.24h._.bmal1ref', 'no_treatment_bmal1._.A02._.24h._.bmal1ref', 'no_treatment_bmal1._.A20._.24h._.bmal1ref', 'no_treatment_bmal1._.A21._.24h._.bmal1ref', 'no_treatment_bmal1._.A22._.24h._.bmal1ref', 'no_treatment_bmal1._.A23._.24h._.bmal1ref', 'no_treatment_bmal1._.A03._.24h._.bmal1ref', 'no_treatment_bmal1._.A04._.24h._.bmal1ref', 'no_treatment_bmal1._.A05._.24h._.bmal1ref', 'no_treatment_bmal1._.A06._.24h._.bmal1ref', 'no_treatment_bmal1._.A07._.24h._.bmal1ref', 'no_treatment_bmal1._.A08._.24h._.bmal1ref', 'no_treatment_bmal1._.A09._.24h._.bmal1ref', 'FBS_bmal

## Final loop

In [8]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/combined_dataset"
concat_df = concat_df[final_order]
display(concat_df)
df_to_treeview(df=concat_df, outFile=out_dir, row_cluster=False, col_cluster=False, sortIndex=False)

,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Radial Mean Ratio SER-Edge,Nuclei - Cell Region TMR/AF555 Rabbitts (global) Threshold Compactness 30%,Nuclei - Cell Region TMR/AF555 Rabbitts (global) SER Hole 2 px,Nuclei - Cell Region TMR/AF555 Rabbitts (global) Symmetry 05 SP-Filter,Nuclei - Ring Region TMR/AF555 Rabbitts (global) Symmetry 15 SP-Filter,Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 03 SP-Filter,Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Symmetry 05 SP-Filter,Nuclei - Ring Region TMR/AF555 Rabbitts (global) Symmetry 05,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Radial Mean Ratio SP-Filter,Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Gabor Min 2 px w2,...,Nuclei - Cell Region Alexa 488 shortened (global) SER Dark 2 px,Nuclei - Cytoplasm Region Alexa 488 shortened (global) Profile 1/5 SER-Hole,Nuclei - Cell Region Alexa 488 shortened (global) SER Hole 2 px,Nuclei - Membrane Region Alexa 488 shortened (global) Radial Mean Ratio SP-Filter,Nuclei - Nucleus Region Alexa 488 shortened (global) Radial Mean Ratio SER-Valley,Nuclei - Nucleus Region Alexa 488 shortened (global) Radial Mean Ratio SER-Dark,Nuclei - Cell Region Alexa 488 shortened (global) Profile 3/5,Nuclei - Cytoplasm Region Alexa 488 shortened (global) Profile 3/5,Nuclei - Membrane Region Alexa 488 shortened (global) Haralick Correlation 2 px,Nuclei - Cytoplasm Region Alexa 488 shortened (global) Haralick Correlation 2 px
human_readable_ids,,,,,,,,,,,,,,,,,,,,,
no_treatment_bmal1._.A10._.24h._.bmal1ref,0.000329,-0.000273,0.000620,0.000145,-0.000131,-0.000363,0.000046,0.000082,0.002384,0.000724,...,0.007912,0.002912,0.001637,-0.000106,0.000039,0.000218,0.002622,0.000873,-0.000722,-0.000760
no_treatment_bmal1._.A11._.24h._.bmal1ref,0.000393,-0.000058,-0.001623,-0.000168,-0.000216,0.000211,-0.000202,-0.000524,-0.000577,0.000735,...,0.000852,-0.000075,0.000447,-0.000250,0.000207,0.000138,0.001785,0.000679,-0.000526,0.000468
no_treatment_bmal1._.A12._.24h._.bmal1ref,0.000328,0.000565,-0.000472,0.000180,-0.000153,0.000213,0.000044,-0.000512,0.000877,0.000285,...,-0.003497,-0.000766,-0.000269,-0.000066,-0.000047,-0.000054,-0.000157,-0.000272,0.002253,0.005417
no_treatment_bmal1._.A13._.24h._.bmal1ref,-0.000804,-0.000175,-0.000081,-0.000115,-0.000151,-0.000234,0.000183,-0.000075,-0.003060,-0.000005,...,0.000731,0.001091,0.000846,-0.000045,0.000126,0.000415,0.002013,0.001461,0.000365,0.000946
no_treatment_bmal1._.A14._.24h._.bmal1ref,-0.001551,0.001929,-0.000009,0.000446,0.000690,0.000179,0.000516,0.000532,0.014712,0.001182,...,-0.014870,-0.007496,-0.001914,-0.000667,-0.000100,-0.000168,-0.009256,-0.014231,0.000483,-0.001161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PF_per2._.O05._.72h._.per2ref,0.000292,0.000610,-0.000165,0.001119,0.001056,0.000137,0.000454,0.000914,0.004755,-0.000168,...,-0.002113,-0.046316,-0.003155,-0.000037,-0.000221,0.000077,-0.008087,-0.000494,-0.018457,-0.023640
PF_per2._.O06._.72h._.per2ref,0.000671,-0.000777,-0.007150,0.000708,0.000633,0.000907,0.001392,0.000512,0.005460,-0.000647,...,-0.007692,-0.043473,-0.007001,-0.000347,0.000767,0.000658,-0.001242,-0.000377,-0.008478,-0.004508
PF_per2._.O07._.72h._.per2ref,0.000275,0.000917,-0.003727,0.000377,0.000498,0.000323,0.000130,0.001650,0.000628,-0.000072,...,-0.013820,-0.039190,-0.008532,-0.000193,-0.000483,0.000124,-0.000167,-0.000240,-0.003019,0.000359


In [9]:
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/feat_red_channel_sorted/"
clean_folder(out_dir)
for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    curr_df = curr_df[final_order]
    out_file = os.path.splitext(file)[0]
    
    df_to_treeview(df=curr_df, outFile=os.path.join(out_dir, out_file), row_cluster=False, col_cluster=False, sortIndex=False)
    

# Split by bmal1 and per2 

## We need the following:

1. all plates/timepoints and treatment in one heatmap (but separate heatmap for Bmal1 and separate heatmap for Per2) - this one is probably the one you want to use for clustering features within channel and using that order in all other heatmaps
2. 'no treatment' heatmap (one for Bmal1 and one for Per2) - only no treatment rows but all timepoints in one heatmap 
3. 'Dex heatmap (one for Bmal1 and one for Per2) - only Dex rows but all timepoints in one heatmap 
4. 'PF' heatmap (one for Bmal1 and one for Per2) - only PF rows but all timepoints in one heatmap 
5. '24h' heatmap (one for Bmal1 and one for Per2) - only '24h' plate but all treatments in one heatmap 
6. '66h' heatmap (one for Bmal1 and one for Per2) - only '66h' plate but all treatments in one heatmap 
7. '72h' heatmap (one for Bmal1 and one for Per2) - only '72h' plate but all treatments in one heatmap


*Note: BMAL1 files use BMAL1ref!*

bmal1

In [10]:
bmal_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/bmal1"
in_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

clean_folder(bmal_out_dir)

bmal1_files = [os.path.join(in_dir, file) for file in os.listdir(in_dir) if 'bmal1ref' in file]

concat_df = [pd.read_csv(file, index_col=0) for file in bmal1_files]
concat_df = pd.concat(concat_df, axis=0)
concat_df = concat_df[concat_df.index.str.contains("_bmal1._.", regex=False)]

# display(concat_df)

# grabbing channels and clusters

channel_dfs = {channel: concat_df[[col for col in concat_df.columns if channel in col]] for channel in channels_found}

bmal_clus_order = list()
channel_out_dir = os.path.join(bmal_out_dir, "channels")
for name, df in channel_dfs.items():
    print(name, df.shape)
    
    fix_name = name.replace("/", "_")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir)
    bmal_clus_order.extend(order)

print(len(bmal_clus_order))
print(bmal_clus_order)

all_timepoints = concat_df[bmal_clus_order]

# combined timepoints
df_to_treeview(df=all_timepoints, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "combined_timepoints"))

# no treatment
no_treatment = all_timepoints[all_timepoints.index.str.contains("no_treatment_", regex=False)]

try:
    df_to_treeview(df=no_treatment, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "no_treatment"))
except:
    pass

# Dex
dex = all_timepoints[all_timepoints.index.str.contains("Dex ", regex=False)]

try:
    df_to_treeview(df=dex, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "Dex"))
except:
    pass

#PF

pf = all_timepoints[all_timepoints.index.str.contains("PF_", regex=False)]

try:
    df_to_treeview(df=pf, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "PF"))
except:
    pass

#24h

h24 = all_timepoints[all_timepoints.index.str.contains("._.24h._.", regex=False)]

try:
    df_to_treeview(df=h24, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "24h"))
except:
    pass

#66h

h66 = all_timepoints[all_timepoints.index.str.contains("._.66h._.", regex=False)]

try:
    df_to_treeview(df=h66, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "66h"))
except:
    pass

#72h

h72 = all_timepoints[all_timepoints.index.str.contains("._.72h._.", regex=False)]

try:
    df_to_treeview(df=h72, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(bmal_out_dir, "72h"))
except:
    pass

TMR/AF555 Rabbitts (264, 162)
SYTO14 Rabbitts (264, 151)
MitoTracker Deep Red (264, 225)
HOECHST 33342 expanded (264, 205)
Alexa 488 shortened (264, 131)
874
['Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Hole', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Saddle', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Valley', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Spot', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Ridge', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Edge', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 15 SP-Filter', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Symmetry 05 SP-Filter', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Symmetry 15', 'Nuclei - Cell Region TMR/AF555 Rabbitts (global) Symmetry 

Per2

In [11]:
per2_out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/per2"
in_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned_renamed"

clean_folder(per2_out_dir)

per2_files = [os.path.join(in_dir, file) for file in os.listdir(in_dir) if 'per2ref' in file]

concat_df = [pd.read_csv(file, index_col=0) for file in per2_files]
concat_df = pd.concat(concat_df, axis=0)
concat_df = concat_df[concat_df.index.str.contains("_per2._.", regex=False)]

# display(concat_df)

# grabbing channels and clusters

channel_dfs = {channel: concat_df[[col for col in concat_df.columns if channel in col]] for channel in channels_found}

per2_clus_order = list()
channel_out_dir = os.path.join(per2_out_dir, "channels")
for name, df in channel_dfs.items():
    print(name, df.shape)
    
    fix_name = name.replace("/", "_")
    order = gen_treeview_channel(df=df, channel_name=fix_name, out_dir=channel_out_dir)
    per2_clus_order.extend(order)

print(len(per2_clus_order))
print(per2_clus_order)

all_timepoints = concat_df[per2_clus_order]

# combined timepoints
df_to_treeview(df=all_timepoints, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "combined_timepoints"))

# no treatment
no_treatment = all_timepoints[all_timepoints.index.str.contains("no_treatment_", regex=False)]

try:
    df_to_treeview(df=no_treatment, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "no_treatment"))
except:
    pass

# Dex
dex = all_timepoints[all_timepoints.index.str.contains("Dex ", regex=False)]

try:
    df_to_treeview(df=dex, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "Dex"))
except:
    pass

#PF

pf = all_timepoints[all_timepoints.index.str.contains("PF_", regex=False)]

try:
    df_to_treeview(df=pf, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "PF"))
except:
    pass

#24h

h24 = all_timepoints[all_timepoints.index.str.contains("._.24h._.", regex=False)]

try:
    df_to_treeview(df=h24, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "24h"))
except:
    pass

#66h

h66 = all_timepoints[all_timepoints.index.str.contains("._.66h._.", regex=False)]

try:
    df_to_treeview(df=h66, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "66h"))
except:
    pass

#72h

h72 = all_timepoints[all_timepoints.index.str.contains("._.72h._.", regex=False)]

try:
    df_to_treeview(df=h72, row_cluster=False, col_cluster=False, sortIndex=False, outFile=os.path.join(per2_out_dir, "72h"))
except:
    pass

TMR/AF555 Rabbitts (264, 162)
SYTO14 Rabbitts (264, 151)
MitoTracker Deep Red (264, 225)
HOECHST 33342 expanded (264, 205)
Alexa 488 shortened (264, 131)
874
['Nuclei - Cell Region TMR/AF555 Rabbitts (global) Profile 2/5', 'Nuclei - Ring Region TMR/AF555 Rabbitts (global) Profile 5/5 SER-Hole', 'Nuclei - Intensity Cytoplasm Region TMR/AF555 Rabbitts (global) Contrast', 'Nuclei - Intensity Membrane Region TMR/AF555 Rabbitts (global) Contrast', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Ridge', 'Nuclei - Membrane Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Spot', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Saddle', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Valley', 'Nuclei - Cytoplasm Region TMR/AF555 Rabbitts (global) Radial Relative Deviation SER-Edge', 'Nuclei - Nucleus Region TMR/AF555 Rabbitts (global) Symmetry 03 SER-Edge', 'Nuclei - Nucl